# 5단계 추가 실험: 시드 2, 3 (Kaggle)

- 시드 1(코랩)과 **같은 학습 계획**(지문 c15466d53a01cedb)을 학습 시드만 바꿔 다시 학습·평가한다
- 평가 시드 12345도 같아서 모든 묶음·시드가 같은 50개 초기 상태에서 평가받는다
- 순서: (선별, 무작위, 전체) × 시드 2, 3 → (프레임 맞춘 무작위, 조건별 균등) × 시드 2, 3. 중간에 끊겨도 핵심 비교가 먼저 남는다
- 결과는 한 번 끝날 때마다 `/kaggle/working/results_seeds.json`에 바로 쓴다 (Output 탭에서 받는다)
- 체크포인트: 학습이 끝난 모델은 평가 전에 `/kaggle/working/ckpt`에 저장. 꺼지면 그 버전 Output을 입력으로 붙이고 다시 실행하면 끝난 것은 건너뛴다 (셀 6-1 설명)
- SMOKE = True면 아주 짧게(2 epoch, 롤아웃 2회) 한 번만 돌려 설치·렌더링·경로를 확인한다
- 본 실행은 Save Version → Save & Run All (브라우저를 닫아도 서버에서 끝까지 돈다)

In [ ]:
# 1. GPU와 파이썬 버전 확인
import sys, torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print('[결과] python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 2. 설치 (코랩 노트북과 같은 고정 버전, 작업 폴더만 /kaggle/temp)
#  - 저장소를 /content 바로 아래에 받으면 코랩이 sys.path에 /content를 넣어 두어서, 설치된 패키지 대신 저장소 폴더를 import 한다 → /content/src 아래에 받는다
#  - robosuite 1.5.1은 mujoco>=3.2.3 만 요구해 최신 3.15가 깔리는데, 관절 처리에서 AssertionError가 난다 → mujoco 3.3.0으로 고정
#  - robomimic은 transformers==4.41.2 등을 고정 요구해 Python 3.13에서 빌드 실패 → 의존성 없이 설치하고 필요한 패키지만 따로 설치
#  - 편집 가능 설치(-e)는 세션 재시작 전까지 import가 안 되므로 일반 설치를 쓴다
import time; t0 = time.time()
WORK = '/kaggle/temp'
%cd {WORK}
!mkdir -p src
![ -d src/robosuite ] || git clone -q https://github.com/ARISE-Initiative/robosuite.git src/robosuite
!cd src/robosuite && git checkout -q v1.5.1
![ -d src/robomimic ] || git clone -q https://github.com/ARISE-Initiative/robomimic.git src/robomimic
!pip install -q ./src/robosuite
!pip install -q --no-deps ./src/robomimic
!pip install -q psutil tqdm termcolor tensorboard tensorboardX imageio imageio-ffmpeg matplotlib egl_probe huggingface_hub transformers diffusers
# robosuite 설치 과정(mink 의존성)에서 mujoco가 최신으로 올라가므로, 마지막에 3.3.0으로 강제로 되돌린다
!pip install -q --no-deps --force-reinstall "mujoco==3.3.0"
import robosuite, robomimic, mujoco
print('[결과] robosuite', robosuite.__version__, '| mujoco', mujoco.__version__, '| robomimic', robomimic.__file__)
print(f'[결과] 설치 {time.time()-t0:.0f}초')

In [ ]:
# 3. 데이터 다운로드 (Can MH low_dim, 107.6MB) 
import os
os.environ['MUJOCO_GL'] = 'egl'   # 화면 없는 서버용 렌더링 설정
os.makedirs(f'{WORK}/data', exist_ok=True)
DATA = f'{WORK}/data/can_mh_low_dim.hdf5'
if not os.path.exists(DATA):
    !wget -q -O {DATA} 'https://huggingface.co/datasets/robomimic/robomimic_datasets/resolve/main/v1.5/can/mh/low_dim_v15.hdf5?download=true'
!ls -la {DATA}

In [ ]:
# 4. 학습 계획 (시드 1과 같은 plan_b64) + 실행 목록
SMOKE = False   # True: 설치·렌더링 확인용 짧은 실행
PLAN_B64 = 'eNrFVstu2zoQ/RetJwBn+PZndBsEhmMrjQHHDmTnArdF/r1nKMqmlRht0QJdGJE4jHg450F+755eu0W3Zu9C2Hi7MrzuN48ddcPbfrndoPblTowENsx3x37Xr093L6vtHjPWh/3T9mu3+N6tdl8PmPm4Xg57rezfXpb962H9fOwWzhjqjqf+9bh87YdxuFuwjj6uTuvn5XH7ra8Dx77HiozFD7vd4e203HcLby6vz4dh++2wrx/tV8Pu/2X/32o3fVVmw5hpzTt1h2HTD93ivlu/DatTv1myN7rF1X5zeKkvxxNK26ctqlfj9eVpWL30R7xjGnb/oGBPR918+83FvSEmIUueEmVivDKxJXbEnjgQJ+JMYkgwT0gsiSPxJIEkkiSSTNaQZbL4jCXryHqygWwkm8hmcoYckxNylpwj58kFcpFcIpcp6aJY1WBZI/hhbYOVDdY1qBVEqCkmBaWoFJbiAiQGJgYiBiQGJgYoFq3hf4GLrW4HcwCNgY0BjoGOAY+BjwGQgZABkZ3uG3MAkYGRAZKBkj3GPca8PmN+wPyAsYC5QbuEuQG1gFpELWK9iFpELaIW8Z2IekQ9oZ6wVsKchHrSFqOeUNdeZNQz6hn1jDUyvpMxL2NOVi5ABnoh2L8oH8qGEoG9ilV+QI3Vd9SxN8HeBHsT7Ec8/nrM8UofnoFZgE+AQ5L+VUIxBhwCHAIMkt3DlfCuFVOVcpaJLdKIjSJGCYzMJ8LqaCb6iDaiiwCBPqKF6CCah76hbegaQKFnaBe6hWYlR4AIhAAIbGgPkKEx6EsOhM5oYybRFMHYKhI0FNJl4BpFEkZxFGFIFUNsBCBVBDMBQKfYO35SxaBk84X8QrhtSJ4IdpXcidiW1FAIFQhfIHoBfjFKrlqNK8kgF/soZGt7C+G2ko75akK1YCGfR/KdHckGdgF2AXYB9gv5dhQA9iBgQUDDWQzgQCKesR/BfgRsSPRVJJiHPV2JBUxUkeCH8Zwf5gG0uB/Vkkq2+JIkY4a0AVKiQ4NiygdViZ9UAolAFrZoIhVB+KIDiMBOOigiyFOYtAHCY3C0IVH4n8KgDYBQDa9QuTG4Gpsrx37kuXDcGPjMa7zmVPnTTZf8nPFXeMMzAqkYFvoTtECcrxzmhrPKV+FFe5crJ+4TTvzDrQNC+fiQ9KnQItW8l0S3JcvPQX6O8FDDG31Cm6qp/eRrVrrU1HC0Kbw55U3tbCcvx4ZAf6buE9pK3k82bvM+zqxsL3bWXC+Uusa+U3Zzk91tbs8tPOW0rTb2MytPGe0q3TKzsa2WNRerAuNIdc3nks1SLdvSHhvLzm2KucHfsGu1KnwiMVWJuCbb21y3jW1RV9lqHBULj3eGf385+Fxf53PDTUdHmrTG5Qyx18dIak4SaTQ4niepKNFMp4qdHyz5Dy4oH4Rr/+JlRW5fWM7nFTdn1vwCwz+5xHBjil+5zLRG+dVLTbi+2KDPH82UbpyLfnY23sjYckbG8ZyU3JyVUg3p6qUp3Lg43TDob52talpXjRsa85rb5+2ViTXnuZ6/9mPmX13a+KcGf3//ASjfygg='
import json, base64, zlib
PLAN = json.loads(zlib.decompress(base64.b64decode(PLAN_B64)))
SETS = {k: [f'demo_{i}' for i in v] for k, v in PLAN['sets'].items()}
SEEDS = [2, 3]
CORE = ['curated_150', 'random_150', 'train']
EXTRA = ['random_frames', 'stratified_random_150']
JOBS = [(n, s) for group in (CORE, EXTRA) for s in SEEDS for n in group]
if SMOKE:
    JOBS = [('curated_150', 2)]
print('[결과] 지문', PLAN['fp'], '| 시드 1 설정', PLAN['config'])
print('[결과] 실행 목록', len(JOBS), JOBS)

In [ ]:
# 5. 묶음별 필터 키를 넣은 학습용 데이터 복사본 (원본은 그대로)
import shutil, h5py, numpy as np
PDATA = f'{WORK}/data/plan.hdf5'
shutil.copy(DATA, PDATA)
with h5py.File(PDATA, 'a') as f:
    for k, ids in SETS.items():
        key = f'mask/{k}'
        if key in f:
            del f[key]
        f.create_dataset(key, data=np.array(ids, dtype='S'))
    print('[결과] 필터 키', {k: len(f[f'mask/{k}']) for k in SETS})

In [ ]:
# 6. 학습·평가 함수
import contextlib, glob, os, time, torch, numpy as np
from robomimic.config import config_factory
from robomimic.scripts.train import train
import robomimic.utils.file_utils as FileUtils
DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EVAL_SEED = 12345  # 모든 묶음이 같은 초기 상태에서 평가받도록 고정

def make_config(cfg, set_name, out_dir, save_every):
    config = config_factory(algo_name='bc')
    with config.values_unlocked():
        config.experiment.name = f"{set_name}_{cfg['algo']}"
        config.train.data = [{'path': PDATA}]
        config.train.output_dir = out_dir
        config.train.hdf5_filter_key = set_name
        config.train.num_epochs = cfg['num_epochs']
        config.experiment.epoch_every_n_steps = cfg['steps_per_epoch']
        config.train.batch_size = cfg['batch_size']
        config.train.seed = cfg['seed']
        config.train.cuda = torch.cuda.is_available()
        config.experiment.validate = False
        config.experiment.rollout.enabled = False   # 평가는 학습 뒤 아래 evaluate()로 따로 한다
        config.experiment.render_video = False
        config.experiment.logging.log_tb = False
        config.experiment.save.enabled = True
        config.experiment.save.every_n_epochs = save_every
        config.experiment.save.on_best_validation = False
        config.experiment.save.on_best_rollout_return = False
        config.experiment.save.on_best_rollout_success_rate = False
        config.observation.modalities.obs.low_dim = ['robot0_eef_pos', 'robot0_eef_quat', 'robot0_gripper_qpos', 'object']
        if cfg['algo'] == 'bc_rnn':   # robomimic 논문의 low_dim BC-RNN 설정
            config.algo.rnn.enabled = True
            config.algo.rnn.horizon = 10
            config.algo.rnn.hidden_dim = 400
            config.algo.rnn.num_layers = 2
            config.train.seq_length = 10
            config.algo.gmm.enabled = True
            config.algo.actor_layer_dims = ()
    return config

def evaluate(ckpt, n, horizon):
    policy, ckpt_dict = FileUtils.policy_from_checkpoint(ckpt_path=ckpt, device=DEV, verbose=False)
    env, _ = FileUtils.env_from_checkpoint(ckpt_dict=ckpt_dict, render=False, render_offscreen=False, verbose=False)
    success = 0
    for i in range(n):
        np.random.seed(EVAL_SEED + i)   # i번째 평가는 모든 묶음에서 같은 캔 초기 위치
        policy.start_episode()
        obs = env.reset()
        for t in range(horizon):
            obs, r, done, _ = env.step(policy(ob=obs))
            if env.is_success()['task']:
                success += 1
                break
    return success

def run_set(set_name, cfg, eval_n):
    # eval_n: {평가할 epoch: 롤아웃 횟수}
    eval_epochs = sorted(eval_n)
    out_dir = f"{WORK}/runs/{set_name}_{cfg['algo']}_s{cfg['seed']}"
    shutil.rmtree(out_dir, ignore_errors=True)
    save_every = eval_epochs[0] if len(eval_epochs) > 1 else cfg['num_epochs']
    t0 = time.time()
    # 학습 로그는 파일로 보낸다 (코랩 출력 창이 길어지면 앞의 결과 줄이 밀려나기 때문)
    os.makedirs(f'{WORK}/logs', exist_ok=True)
    with open(f"{WORK}/logs/{set_name}_{cfg['algo']}.log", 'w') as lf, contextlib.redirect_stdout(lf), contextlib.redirect_stderr(lf):
        train(make_config(cfg, set_name, out_dir, save_every), device=DEV)
    train_sec = time.time() - t0
    rows = {}
    for ep in eval_epochs:
        ckpt = sorted(glob.glob(f'{out_dir}/**/models/model_epoch_{ep}.pth', recursive=True))[-1]
        t1 = time.time()
        rows[ep] = {'success': evaluate(ckpt, eval_n[ep], cfg['rollout_horizon']), 'rollouts': eval_n[ep], 'eval_sec': round(time.time() - t1)}
    return train_sec, rows
print('[결과] 함수 준비, 장치', DEV)

In [ ]:
# 6-1. 체크포인트: 노트북이 중간에 꺼져도 이어서 돌릴 수 있게 한다
#  - 결과: (묶음, 시드) 하나 끝날 때마다 results_seeds.json에 저장 → 다시 돌리면 끝난 것은 건너뜀
#  - 모델: 학습이 끝나면 평가 전에 200·400 epoch 모델을 /kaggle/working/ckpt에 복사
#          → 평가 도중 꺼지면 다시 학습하지 않고 저장된 모델로 평가만 한다
#  - 학습 도중 꺼진 실행은 처음부터 다시 학습한다. 중간 epoch에서 이어 학습하면 데이터 섞는 순서(난수 상태)가
#    달라져 '같은 시드'라는 조건이 깨지기 때문 (손해는 최대 학습 1번, 약 10분)
#  - 이어서 돌리는 법: 꺼진 버전의 Output을 노트북 입력(Add Input → Your Work)으로 붙이고 다시 Save & Run All
CKPT = '/kaggle/working/ckpt'

def find_ckpt(set_name, seed, ep):
    # 이번 세션(/kaggle/working) 또는 입력으로 붙인 이전 버전(/kaggle/input) 에서 찾는다
    pats = [f'{CKPT}/{set_name}_s{seed}/model_epoch_{ep}.pth', f'/kaggle/input/**/ckpt/{set_name}_s{seed}/model_epoch_{ep}.pth']
    for p in pats:
        hit = sorted(glob.glob(p, recursive=True))
        if hit:
            return hit[-1]
    return None

def run_set(set_name, cfg, eval_n):
    eval_epochs = sorted(eval_n)
    seed = cfg['seed']
    ckpts = {ep: find_ckpt(set_name, seed, ep) for ep in eval_epochs}
    train_sec = 0.0
    if not all(ckpts.values()):
        out_dir = f"{WORK}/runs/{set_name}_{cfg['algo']}_s{seed}"
        shutil.rmtree(out_dir, ignore_errors=True)
        save_every = eval_epochs[0] if len(eval_epochs) > 1 else cfg['num_epochs']
        t0 = time.time()
        os.makedirs(f'{WORK}/logs', exist_ok=True)
        with open(f"{WORK}/logs/{set_name}_s{seed}.log", 'w') as lf, contextlib.redirect_stdout(lf), contextlib.redirect_stderr(lf):
            train(make_config(cfg, set_name, out_dir, save_every), device=DEV)
        train_sec = time.time() - t0
        dst = f'{CKPT}/{set_name}_s{seed}'
        os.makedirs(dst, exist_ok=True)
        for ep in eval_epochs:
            src = sorted(glob.glob(f'{out_dir}/**/models/model_epoch_{ep}.pth', recursive=True))[-1]
            shutil.copy(src, f'{dst}/model_epoch_{ep}.pth')
            ckpts[ep] = f'{dst}/model_epoch_{ep}.pth'
        json.dump({'train_sec': round(train_sec)}, open(f'{dst}/train.json', 'w'))
    else:
        meta = glob.glob(f'{CKPT}/{set_name}_s{seed}/train.json') + glob.glob(f'/kaggle/input/**/ckpt/{set_name}_s{seed}/train.json', recursive=True)
        train_sec = json.load(open(meta[0]))['train_sec'] if meta else 0
        print('[체크포인트] 저장된 모델로 평가만', set_name, seed, flush=True)
    rows = {}
    for ep in eval_epochs:
        t1 = time.time()
        rows[ep] = {'success': evaluate(ckpts[ep], eval_n[ep], cfg['rollout_horizon']), 'rollouts': eval_n[ep], 'eval_sec': round(time.time() - t1)}
    return train_sec, rows
print('[결과] 체크포인트 폴더', CKPT)

In [ ]:
# 7. 실행: (묶음, 시드) 하나 끝날 때마다 결과 파일에 바로 쓴다
import datetime, os
OUT = '/kaggle/working/results_smoke.json' if SMOKE else '/kaggle/working/results_seeds.json'
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
doc = json.load(open(OUT)) if os.path.exists(OUT) else {
    'fingerprint': PLAN['fp'], 'run_id': PLAN['run_id'], 'mode': 'smoke' if SMOKE else 'seeds',
    'platform': 'kaggle', 'gpu': gpu, 'eval_seed': EVAL_SEED, 'config': PLAN['config'], 'runs': {}}
# 이전 버전 결과를 입력으로 붙였으면 끝난 실행을 이어받는다
for prev in glob.glob('/kaggle/input/**/' + os.path.basename(OUT), recursive=True):
    old = json.load(open(prev))
    if old.get('fingerprint') == PLAN['fp']:
        for k, v in old['runs'].items():
            doc['runs'].setdefault(k, v)
        print('[체크포인트] 이전 결과 이어받음', prev, len(old['runs']), '개')
for name, seed in JOBS:
    key = f'{name}|seed{seed}'
    if key in doc['runs']:
        continue
    cfg = dict(PLAN['config'], seed=seed)
    if SMOKE:
        cfg.update(num_epochs=2, early_eval_epoch=1)
        eval_n = {1: 1, 2: 2}
    else:
        eval_n = {cfg['num_epochs']: cfg['rollout_n'], cfg['early_eval_epoch']: cfg['early_eval_n']}
    train_sec, rows = run_set(name, cfg, eval_n)
    fin = rows[cfg['num_epochs']]
    doc['runs'][key] = {'set': name, 'seed': seed, 'success': fin['success'], 'rollouts': fin['rollouts'],
                        'train_sec': round(train_sec), 'eval_sec': sum(r['eval_sec'] for r in rows.values()),
                        'by_epoch': {str(k): v for k, v in rows.items()},
                        'finished_at': datetime.datetime.now().isoformat(timespec='seconds')}
    with open(OUT + '.tmp', 'w') as f:
        json.dump(doc, f)
    os.replace(OUT + '.tmp', OUT)
    print('RESULT', key, json.dumps(doc['runs'][key]), flush=True)
print('[결과] 완료', len(doc['runs']), '개 →', OUT)

### 환경 차이 확인 (선택)
시드 1(코랩)과 같은 계획·시드를 이 환경에서 다시 돌려 결과가 같은지 본다. 셀 7을 시드 1, 핵심 3개 묶음으로 바꿔 실행하고 `results_seed1_kaggle.json`에 따로 저장한다.
2026-10-12 실행 결과: 선별 47/50, 무작위 40/50, 전체 48/50 (200 epoch 22/30, 9/30, 21/30) 으로 코랩과 완전히 같았다.

In [ ]:
# 8. (선택) 시드 1 재실행: 필요할 때만 RUN_SEED1 = True
RUN_SEED1 = False
if RUN_SEED1:
    src = [c for c in In if c.startswith('# 7.')][-1]
    src = src.replace("'/kaggle/working/results_seeds.json'", "'/kaggle/working/results_seed1_kaggle.json'")
    src = src.replace('for name, seed in JOBS:', 'for name, seed in [(n, 1) for n in CORE]:')
    get_ipython().run_cell(src)